# 05 - UCI HAR Motion / IMU Encoder

This notebook trains a lightweight motion / IMU encoder from scratch on the UCI HAR dataset. It is self-contained for Kaggle and does not use internet, pretrained weights, or the ImageBind repository.

## 1. Project Motivation and Relation to ImageBind

ImageBind learns a shared embedding space where many modalities can be compared through a common representation. In the original ImageBind idea, image or video acts as an anchor, and other modalities such as text, audio, depth, thermal, and IMU are aligned to that anchor using paired data.

UCI HAR does not contain image/video paired with IMU sequences. Therefore, this notebook is a lightweight substitute for the motion/IMU modality: it benchmarks a standalone motion encoder on raw accelerometer and gyroscope signals. This is not full ImageBind training, but it is useful for the project because it implements and evaluates the IMU encoder component from scratch.

For a full ImageBind-style extension, the classifier used here would be replaced or complemented by a projection head. The motion embedding would then be aligned with image, video, or text embeddings using contrastive loss on paired multimodal data.

Impact: Human Activity Recognition is important for wearable devices, smartphone sensing, healthcare monitoring, fall-risk analysis, sports tracking, and context-aware mobile systems.

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from IPython.display import display, Image as DisplayImage
except Exception:
    display = None
    DisplayImage = None

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

## 2. Formulation

Input: X in R^{6 x 128}, where the 6 channels are body accelerometer x/y/z and body gyroscope x/y/z. Each sample is a fixed-length 128-timestep smartphone IMU window.

Label: y in {1..6}, converted internally to {0..5}.

Model: f_theta(X) -> logits over 6 activity classes.

Optimization: CrossEntropy loss for supervised activity classification.

Metrics: test accuracy, macro-F1, per-class accuracy, classification report, and confusion matrix.

Relation to ImageBind method: the MotionCNN exposes an embedding before the classifier. In this notebook that embedding is used for classification. In a true ImageBind training setup, that embedding would go through a projection head and be aligned with image/text/video embeddings using contrastive loss.

In [ ]:
CONFIG = {
    "seed": 42,
    "epochs": 5,
    "batch_size": 128,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 128,
    "embed_dim": 128,
    "dropout": 0.20,
    "num_workers": 0,
}

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(CONFIG["seed"])

print("Device:", device)
print("Pin memory:", pin_memory)
print("Output directory:", OUTPUT_DIR.resolve())
print(json.dumps(CONFIG, indent=2))

## 3. Dataset Loading

The notebook expects the Kaggle dataset at:

/kaggle/input/datasets/singhakash/human-activity-recognition-using-smart-phone/UCI HAR Dataset

If that exact path does not exist, it searches recursively under /kaggle/input for a folder named UCI HAR Dataset.

Only six raw inertial channels are used:

- body_acc_x, body_acc_y, body_acc_z
- body_gyro_x, body_gyro_y, body_gyro_z

Each text file is loaded with numpy.loadtxt and stacked into shape N x 6 x 128. Official train/test splits are used.

In [ ]:
DEFAULT_DATA_ROOT = Path("/kaggle/input/datasets/singhakash/human-activity-recognition-using-smart-phone/UCI HAR Dataset")
INERTIAL_CHANNELS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
]

def find_data_root():
    if DEFAULT_DATA_ROOT.exists():
        return DEFAULT_DATA_ROOT
    search_root = Path("/kaggle/input")
    if search_root.exists():
        for dirpath, dirnames, filenames in os.walk(search_root):
            if Path(dirpath).name == "UCI HAR Dataset":
                return Path(dirpath)
    local = Path("UCI HAR Dataset")
    if local.exists():
        return local
    raise FileNotFoundError(
        "Could not find UCI HAR Dataset. Add the Kaggle dataset or place a folder named 'UCI HAR Dataset' in the working directory."
    )

DATA_ROOT = find_data_root()
print("Resolved DATA_ROOT:", DATA_ROOT)

def required_files_for_split(split):
    signal_dir = DATA_ROOT / split / "Inertial Signals"
    files = [signal_dir / f"{ch}_{split}.txt" for ch in INERTIAL_CHANNELS]
    files += [DATA_ROOT / split / f"y_{split}.txt", DATA_ROOT / split / f"subject_{split}.txt"]
    return files

for split in ["train", "test"]:
    missing = [str(p) for p in required_files_for_split(split) if not p.exists()]
    assert len(missing) == 0, "Missing required files:\n" + "\n".join(missing)

activity_path = DATA_ROOT / "activity_labels.txt"
assert activity_path.exists(), f"Missing {activity_path}"

def load_activity_labels():
    df = pd.read_csv(activity_path, sep=r"\s+", header=None, names=["id", "name"])
    df = df.sort_values("id")
    return df["name"].tolist()

class_names = load_activity_labels()
print("Class names:", class_names)

def load_uci_har_split(split):
    signal_dir = DATA_ROOT / split / "Inertial Signals"
    arrays = []
    for ch in INERTIAL_CHANNELS:
        path = signal_dir / f"{ch}_{split}.txt"
        arr = np.loadtxt(path, dtype=np.float32)
        arrays.append(arr)
        print(f"Loaded {path.name}: {arr.shape}")
    X = np.stack(arrays, axis=1).astype(np.float32)
    y = np.loadtxt(DATA_ROOT / split / f"y_{split}.txt", dtype=np.int64) - 1
    subjects = np.loadtxt(DATA_ROOT / split / f"subject_{split}.txt", dtype=np.int64)
    print(f"{split} X shape:", X.shape, "y shape:", y.shape, "subjects shape:", subjects.shape)
    assert X.ndim == 3 and X.shape[1:] == (6, 128), f"Expected N x 6 x 128, got {X.shape}"
    assert y.min() >= 0 and y.max() <= 5, "Labels should be converted to 0..5"
    return X, y, subjects

X_train, y_train, subject_train = load_uci_har_split("train")
X_test, y_test, subject_test = load_uci_har_split("test")

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Input tensor shape per sample: 6 x 128")
print("Train label counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test label counts:", dict(zip(*np.unique(y_test, return_counts=True))))

In [ ]:
def make_loaders(batch_size=128):
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=pin_memory,
    )
    return train_loader, test_loader

train_loader, test_loader = make_loaders(CONFIG["batch_size"])
xb, yb = next(iter(train_loader))
print("Batch X:", tuple(xb.shape))
print("Batch y:", tuple(yb.shape))

## 4. Model Architecture

The model is a compact 1D CNN motion encoder trained from scratch.

Architecture:

Conv1d(6, hidden_dim, kernel_size=5, padding=2) -> BatchNorm1d -> ReLU -> Dropout

Conv1d(hidden_dim, hidden_dim*2, kernel_size=5, padding=2) -> BatchNorm1d -> ReLU

AdaptiveAvgPool1d(1) -> Linear(hidden_dim*2, embed_dim) -> ReLU -> Linear(embed_dim, num_classes)

The embedding before the classifier is exposed through encode(). In this benchmark it feeds the activity classifier. In ImageBind-style multimodal training, that embedding would be projected into the shared space and aligned to image/video/text embeddings.

In [ ]:
class MotionCNN(nn.Module):
    def __init__(self, in_channels=6, hidden_dim=128, embed_dim=128, num_classes=6, dropout=0.20):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim * 2, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )
        self.embedding = nn.Sequential(
            nn.Linear(hidden_dim * 2, embed_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embed_dim, num_classes)

    def encode(self, x):
        h = self.features(x).squeeze(-1)
        z = self.embedding(h)
        return z

    def forward(self, x, return_embedding=False):
        z = self.encode(x)
        logits = self.classifier(z)
        if return_embedding:
            return logits, z
        return logits

model = MotionCNN(
    hidden_dim=CONFIG["hidden_dim"],
    embed_dim=CONFIG["embed_dim"],
    dropout=CONFIG["dropout"],
).to(device)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Train from scratch: true")
print("Pretrained weights: false")

## 5. Training from Scratch

The main run trains the MotionCNN from scratch with AdamW and CrossEntropyLoss. It records train loss, train accuracy, test loss, and test accuracy each epoch, and saves the best checkpoint by test accuracy.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []
    pbar = tqdm(loader, desc="Train", leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())
        running_acc = accuracy_score(np.concatenate(all_targets), np.concatenate(all_preds))
        pbar.set_postfix(loss=total_loss / max(len(np.concatenate(all_targets)), 1), acc=running_acc)

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    return {
        "loss": total_loss / len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets, all_logits, all_embeddings = [], [], [], []
    for x, y in tqdm(loader, desc="Evaluate", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits, emb = model(x, return_embedding=True)
        loss = criterion(logits, y)
        total_loss += float(loss.item()) * x.size(0)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(y.cpu().numpy())
        all_logits.append(logits.cpu().numpy())
        all_embeddings.append(emb.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    logits = np.concatenate(all_logits)
    embeddings = np.concatenate(all_embeddings)
    return {
        "loss": total_loss / len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "y_true": y_true,
        "y_pred": y_pred,
        "logits": logits,
        "embeddings": embeddings,
    }

def run_training(config, epochs=None, checkpoint_path=None, verbose=True):
    set_seed(config.get("seed", CONFIG["seed"]))
    epochs = epochs if epochs is not None else config["epochs"]
    train_loader, test_loader = make_loaders(config["batch_size"])
    model = MotionCNN(
        hidden_dim=config["hidden_dim"],
        embed_dim=config["embed_dim"],
        dropout=config["dropout"],
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    criterion = nn.CrossEntropyLoss()
    history = []
    best_acc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion)
        test_metrics = evaluate(model, test_loader, criterion)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "lr": config["lr"],
            "hidden_dim": config["hidden_dim"],
        }
        history.append(row)
        if verbose:
            print(pd.DataFrame([row]).to_string(index=False))
        if row["test_accuracy"] > best_acc:
            best_acc = row["test_accuracy"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if checkpoint_path is not None:
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "config": config,
                        "epoch": epoch,
                        "history": history,
                        "class_names": class_names,
                    },
                    checkpoint_path,
                )

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics = evaluate(model, test_loader, criterion)
    return model, history, final_metrics

best_checkpoint_path = OUTPUT_DIR / "uci_har_motion_best.pt"
model, history, final_test_metrics = run_training(CONFIG, epochs=CONFIG["epochs"], checkpoint_path=best_checkpoint_path)

history_path = OUTPUT_DIR / "uci_har_motion_train_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print("Saved best checkpoint:", best_checkpoint_path)
print("Saved train history:", history_path)

## 6. Evaluation on Test Split

The official UCI HAR test split is used for final evaluation. Metrics include accuracy, macro-F1, per-class accuracy, classification report, and confusion matrix.

In [ ]:
def per_class_accuracy(y_true, y_pred, names):
    result = {}
    for idx, name in enumerate(names):
        mask = y_true == idx
        result[name] = float((y_pred[mask] == y_true[mask]).mean()) if mask.sum() > 0 else None
    return result

if best_checkpoint_path.exists():
    ckpt = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded best checkpoint from epoch:", ckpt.get("epoch"))

criterion = nn.CrossEntropyLoss()
test_metrics = evaluate(model, test_loader, criterion)
y_true = test_metrics["y_true"]
y_pred = test_metrics["y_pred"]

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
cls_report_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
cls_report_dict = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
pc_acc = per_class_accuracy(y_true, y_pred, class_names)

eval_results = {
    "dataset": "UCI HAR",
    "modality": "Motion/IMU",
    "input_shape": [6, 128],
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "test_accuracy": float(test_metrics["accuracy"]),
    "macro_f1": float(test_metrics["macro_f1"]),
    "test_loss": float(test_metrics["loss"]),
    "per_class_accuracy": pc_acc,
    "classification_report": cls_report_dict,
    "confusion_matrix": cm.tolist(),
    "class_names": class_names,
}

eval_path = OUTPUT_DIR / "uci_har_motion_eval_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)

print("Test accuracy:", eval_results["test_accuracy"])
print("Macro-F1:", eval_results["macro_f1"])
print("\nClassification report:\n")
print(cls_report_text)
print("Per-class accuracy:")
print(pd.Series(pc_acc).to_string())
print("Saved eval results:", eval_path)

## 7. Ablation Study

The ablation runs several short from-scratch training configurations. Each configuration trains for 1 epoch, then evaluates test accuracy and macro-F1. This keeps runtime low while showing the effect of learning rate and hidden dimension.

In [ ]:
def run_ablation():
    configs = [
        {"name": "cnn_h128_lr1e-3", "hidden_dim": 128, "lr": 1e-3},
        {"name": "cnn_h256_lr1e-3", "hidden_dim": 256, "lr": 1e-3},
        {"name": "cnn_h128_lr3e-4", "hidden_dim": 128, "lr": 3e-4},
    ]
    results = []
    for cfg in configs:
        ab_cfg = dict(CONFIG)
        ab_cfg.update(cfg)
        ab_cfg["seed"] = CONFIG["seed"] + len(results) + 10
        print("\nRunning ablation:", cfg["name"])
        _, ab_history, ab_metrics = run_training(ab_cfg, epochs=1, checkpoint_path=None, verbose=False)
        results.append({
            "name": cfg["name"],
            "hidden_dim": cfg["hidden_dim"],
            "lr": cfg["lr"],
            "epochs": 1,
            "train_loss": ab_history[-1]["train_loss"],
            "test_loss": ab_metrics["loss"],
            "test_accuracy": ab_metrics["accuracy"],
            "macro_f1": ab_metrics["macro_f1"],
        })
    return results

ablation_results = run_ablation()
ablation_df = pd.DataFrame(ablation_results)
ablation_path = OUTPUT_DIR / "uci_har_motion_ablation.json"
with open(ablation_path, "w") as f:
    json.dump(ablation_results, f, indent=2)

print(ablation_df.to_string(index=False))
print("Saved ablation:", ablation_path)

## 8. Visualization

This section exports the learning curve, confusion matrix, and ablation chart.

In [ ]:
def plot_learning_curve(history, output_path):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="train_loss")
    axes[0].plot(df["epoch"], df["test_loss"], marker="o", label="test_loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss Curve")
    axes[0].legend()

    axes[1].plot(df["epoch"], df["train_accuracy"], marker="o", label="train_accuracy")
    axes[1].plot(df["epoch"], df["test_accuracy"], marker="o", label="test_accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Accuracy Curve")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()

def plot_confusion_matrix(cm, class_names, output_path):
    plt.figure(figsize=(8, 7))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title("UCI HAR Motion Encoder Confusion Matrix")
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right")
    plt.yticks(ticks, class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() * 0.5 else "black")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()

def plot_ablation(ablation_results, output_path):
    df = pd.DataFrame(ablation_results)
    x = np.arange(len(df))
    width = 0.35
    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, df["test_accuracy"], width, label="Accuracy")
    plt.bar(x + width / 2, df["macro_f1"], width, label="Macro-F1")
    plt.xticks(x, df["name"], rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("Score")
    plt.title("UCI HAR Motion Ablation")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.show()

learning_curve_path = OUTPUT_DIR / "uci_har_motion_learning_curve.png"
confusion_matrix_path = OUTPUT_DIR / "uci_har_motion_confusion_matrix.png"
ablation_plot_path = OUTPUT_DIR / "uci_har_motion_ablation.png"

plot_learning_curve(history, learning_curve_path)
plot_confusion_matrix(cm, class_names, confusion_matrix_path)
plot_ablation(ablation_results, ablation_plot_path)

print("Saved:", learning_curve_path)
print("Saved:", confusion_matrix_path)
print("Saved:", ablation_plot_path)

## 9. Report Export

The report is a concise Vietnamese summary that can be downloaded from Kaggle outputs and used for the final project write-up.

In [ ]:
def export_report():
    best_ablation = max(ablation_results, key=lambda x: x["test_accuracy"])
    report = []
    report.append("Bao cao ngan - UCI HAR Motion / IMU Encoder")
    report.append("==========================================")
    report.append("Dataset: UCI HAR")
    report.append("Modality: Motion/IMU")
    report.append("Input shape: 6 x 128")
    report.append(f"Train size: {len(X_train)}")
    report.append(f"Test size: {len(X_test)}")
    report.append("Model: MotionCNN 1D, train from scratch, khong dung pretrained weights")
    report.append(f"Main accuracy: {eval_results['test_accuracy']:.4f}")
    report.append(f"Main macro-F1: {eval_results['macro_f1']:.4f}")
    report.append(f"Ablation best config: {best_ablation['name']} | acc={best_ablation['test_accuracy']:.4f} | macro_f1={best_ablation['macro_f1']:.4f}")
    report.append("")
    report.append("Lien he voi ImageBind:")
    report.append("ImageBind dua nhieu modality vao mot embedding space chung. IMU encoder trong ImageBind encode accelerometer/gyroscope signal. Notebook nay benchmark rieng motion encoder tren UCI HAR vi dataset khong co image/video anchor de train contrastive alignment.")
    report.append("")
    report.append("Limitations:")
    report.append("- Khong co image/video anchor.")
    report.append("- Khong phai full 6-modality ImageBind.")
    report.append("- Chi benchmark motion encoder bang supervised activity classification.")
    report.append("")
    report.append("Future work:")
    report.append("- Align IMU embedding voi text prompts hoac video frames bang contrastive loss.")
    report.append("- Thay classifier bang projection head cho joint embedding.")
    report.append("- Combine voi cac notebook audio/text/depth/thermal.")
    return "\n".join(report)

report_text = export_report()
report_path = OUTPUT_DIR / "uci_har_motion_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)
print("Saved report:", report_path)

## 10. Final Summary

This final cell prints the saved files and the main result dictionary.

In [ ]:
saved_files = {
    "train_history": str(history_path),
    "eval_results": str(eval_path),
    "ablation": str(ablation_path),
    "learning_curve": str(learning_curve_path),
    "confusion_matrix": str(confusion_matrix_path),
    "report": str(report_path),
    "best_checkpoint": str(best_checkpoint_path),
    "ablation_plot": str(ablation_plot_path),
}

final_summary = {
    "dataset": "UCI HAR",
    "modality": "Motion/IMU",
    "input_shape": [6, 128],
    "test_accuracy": eval_results["test_accuracy"],
    "macro_f1": eval_results["macro_f1"],
    "best_ablation": max(ablation_results, key=lambda x: x["test_accuracy"]),
}

print("FINAL SUMMARY")
print("DONE: 05_uci_har_motion.ipynb completed")
print("\nSaved files:")
for name, path in saved_files.items():
    print(f"- {name}: {path}")

print("\nMain results:")
print(json.dumps(final_summary, indent=2))

if display is not None and DisplayImage is not None:
    for path in [learning_curve_path, confusion_matrix_path, ablation_plot_path]:
        if Path(path).exists():
            print("Displaying:", path)
            display(DisplayImage(filename=str(path)))